# HW01-B — SQL, Latency, and Metabase

The business team does not care that your notebook works. They want a dashboard that opens fast.

Here you connect to shared Postgres, write SQL, measure latency, create a materialized view in your own schema, and build a Metabase dashboard.

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials. Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

- PostgreSQL `EXPLAIN`: https://www.postgresql.org/docs/current/sql-explain.html
- PostgreSQL using `EXPLAIN`: https://www.postgresql.org/docs/current/using-explain.html
- Metabase questions: https://www.metabase.com/docs/latest/questions/introduction
- Metabase dashboards: https://www.metabase.com/docs/latest/dashboards/introduction

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

## What to avoid

- `select *` in dashboard queries.
- Creating objects in `core`. You do not own `core`.
- Optimizing without runtime measurements.
- Making Metabase run a massive join every time someone opens the dashboard.

In [ ]:
import os, re, time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

for path in ['sql', 'reports', 'screenshots']:
    Path(path).mkdir(exist_ok=True)

DB_HOST     = os.getenv('QBC12_DB_HOST', '185.50.38.163')  # ← تغییر داده شد
DB_PORT     = os.getenv('QBC12_DB_PORT', '32112')
DB_NAME     = os.getenv('QBC12_DB_NAME', 'qbc12_airbnb')
DB_USER     = os.getenv('QBC12_DB_USER', '') or input('DB user: ').strip()
DB_PASSWORD = os.getenv('QBC12_DB_PASSWORD', '') or input('DB password: ').strip()
STUDENT_ID  = os.getenv('QBC12_STUDENT_ID', '') or DB_USER.replace('student_', '')

safe_student   = re.sub(r'[^a-zA-Z0-9_]', '_', STUDENT_ID.lower())
STUDENT_SCHEMA = f'student_{safe_student}'
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    pool_pre_ping=True
)
with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = '30s'"))
    version = conn.execute(text('select version()')).scalar()

print(STUDENT_SCHEMA)
print(version[:80])

## 1. Inspect before querying

You are not allowed to write the final query blind. Check columns and row counts first.

In [21]:
# Section 1: columns
columns_sql = """
SELECT table_schema, table_name, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'core'
  AND table_name IN ('listing', 'calendar_day', 'review')
ORDER BY table_name, ordinal_position;
"""
with engine.connect() as conn:
    df_cols = pd.read_sql(text(columns_sql), conn)
df_cols

,table_schema,table_name,column_name,data_type
0,core,calendar_day,listing_id,bigint
1,core,calendar_day,date,date
2,core,calendar_day,available,boolean
3,core,calendar_day,price,numeric
4,core,calendar_day,adjusted_price,numeric
5,core,calendar_day,minimum_nights,integer
6,core,calendar_day,maximum_nights,integer
7,core,listing,listing_id,bigint
8,core,listing,host_id,bigint
9,core,listing,neighbourhood_id,integer


In [23]:
# Section 1: row counts
row_count_sql = """
SELECT 'core.listing'      AS table_name, COUNT(*) AS rows FROM core.listing
UNION ALL
SELECT 'core.calendar_day',               COUNT(*)          FROM core.calendar_day
UNION ALL
SELECT 'core.review',                     COUNT(*)          FROM core.review;
"""
with engine.connect() as conn:
    df_counts = pd.read_sql(text(row_count_sql), conn)
df_counts

,table_name,rows
0,core.listing,10480
1,core.calendar_day,3825200
2,core.review,501084


## 2. Create your sandbox schema

This is the only place you write database objects.

In [ ]:
# TODO 2.1
# Create your schema if it does not exist.
# Schema name is STUDENT_SCHEMA.

# Write your code here.

In [27]:
from sqlalchemy import text

with engine.connect() as conn:
    rows = conn.execute(text("""
        select schema_name
        from information_schema.schemata
        where schema_name = :schema_name
    """), {"schema_name": STUDENT_SCHEMA}).fetchall()

print(rows)

[('student_nazanin_hesari',)]


In [31]:
print(f"Using existing schema: {STUDENT_SCHEMA}")

Using existing schema: student_nazanin_hesari


## 3. Build baseline SQL in pieces

Do not write one giant query first. Build the CTEs, test them, then combine.

In [ ]:
# TODO 3.1
# Write calendar_30_sql.
# Required output: listing_id, avg_calendar_price_30, availability_30_rate.

calendar_30_sql = '''
-- write SQL here
'''

In [33]:
calendar_30_sql = """
SELECT
    listing_id,
    AVG(price)                                                          AS avg_calendar_price_30,
    ROUND(
        SUM(CASE WHEN available = true THEN 1 ELSE 0 END)::numeric
        / NULLIF(COUNT(*), 0)::numeric, 4
    )                                                                   AS availability_30_rate
FROM core.calendar_day
WHERE date >= (SELECT MIN(date) FROM core.calendar_day)
  AND date <  (SELECT MIN(date) FROM core.calendar_day) + INTERVAL '30 days'
GROUP BY listing_id
"""
with engine.connect() as conn:
    pd.read_sql(text(calendar_30_sql), conn).head()

In [ ]:
# TODO 3.2
# Write review_counts_sql.
# Required output: listing_id, total_reviews.

review_counts_sql = '''
-- write SQL here
'''

In [35]:
review_counts_sql = """
SELECT
    listing_id,
    COUNT(*) AS total_reviews
FROM core.review
GROUP BY listing_id
"""
with engine.connect() as conn:
    pd.read_sql(text(review_counts_sql), conn).head()

In [ ]:
# TODO 3.3
# Combine the CTEs with core.listing into baseline_sql.
# Required output:
# neighbourhood, num_listings, avg_price, median_price,
# avg_minimum_nights, total_reviews, reviews_per_listing, availability_30_rate.

baseline_sql = '''
-- write SQL here
'''
Path('sql/01_baseline_neighbourhood_summary.sql').write_text(baseline_sql)

In [39]:
cols_sql = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'core'
  AND table_name = 'listing'
ORDER BY ordinal_position;
"""
with engine.connect() as conn:
    df_cols = pd.read_sql(text(cols_sql), conn)

df_cols

,column_name,data_type
0,listing_id,bigint
1,host_id,bigint
2,neighbourhood_id,integer
3,room_type,text
4,property_type,text
5,accommodates,integer
6,bedrooms,numeric
7,beds,numeric
8,bathrooms_text,text
9,listing_price,numeric


In [55]:
baseline_sql = """
WITH calendar_30 AS (
    SELECT
        cd.listing_id,
        AVG(cd.price) AS avg_calendar_price_30,
        ROUND(
            SUM(CASE WHEN cd.available = true THEN 1 ELSE 0 END)::numeric
            / NULLIF(COUNT(*), 0)::numeric,
            4
        ) AS availability_30_rate
    FROM core.calendar_day cd
    WHERE cd.date >= (SELECT MIN(date) FROM core.calendar_day)
      AND cd.date <  (SELECT MIN(date) FROM core.calendar_day) + INTERVAL '30 days'
    GROUP BY cd.listing_id
),
review_counts AS (
    SELECT
        r.listing_id,
        COUNT(*) AS total_reviews
    FROM core.review r
    GROUP BY r.listing_id
)
SELECT
    l.neighbourhood_id::text AS neighbourhood,
    COUNT(l.listing_id) AS num_listings,
    ROUND(AVG(l.listing_price), 2) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
    ROUND(AVG(l.minimum_nights::numeric), 2) AS avg_minimum_nights,
    COALESCE(SUM(rc.total_reviews), 0) AS total_reviews,
    ROUND(
        COALESCE(SUM(rc.total_reviews), 0)::numeric
        / NULLIF(COUNT(l.listing_id), 0)::numeric,
        2
    ) AS reviews_per_listing,
    ROUND(AVG(c30.availability_30_rate), 4) AS availability_30_rate
FROM core.listing l
LEFT JOIN calendar_30 c30
    ON c30.listing_id = l.listing_id
LEFT JOIN review_counts rc
    ON rc.listing_id = l.listing_id
GROUP BY l.neighbourhood_id
ORDER BY num_listings DESC
"""

In [57]:
Path("sql/01_baseline_neighbourhood_summary.sql").write_text(baseline_sql)

1404

In [59]:
from sqlalchemy import text
import time
import pandas as pd

def timed_read_sql(sql: str, repeats: int = 3):
    times = []
    last_df = None
    for _ in range(repeats):
        start = time.perf_counter()
        with engine.connect() as conn:
            last_df = pd.read_sql(text(sql), conn)
        times.append(time.perf_counter() - start)
    return last_df, times

In [61]:
baseline_df, baseline_times = timed_read_sql(baseline_sql, repeats=3)
baseline_df.head(), baseline_times

(  neighbourhood  num_listings  avg_price  median_price  avg_minimum_nights  \
 0            22          1808     271.28         240.0                3.95   
 1             2          1207     315.88         245.5                4.01   
 2            20          1199     280.47         250.0                5.46   
 3             1           923     307.72         240.0                4.89   
 4             4           736     255.04         214.5                4.24   
 
    total_reviews  reviews_per_listing  availability_30_rate  
 0        62753.0                34.71                0.1554  
 1       106496.0                88.23                0.2189  
 2        45931.0                38.31                0.1703  
 3        76899.0                83.31                0.2214  
 4        26668.0                36.23                0.1390  ,
 [0.4480480000056559, 0.48890949999622535, 0.4813768999883905])

## 4. Read the query plan

`EXPLAIN ANALYZE` actually runs the query. Look for big scans, expensive joins, and repeated work.

In [ ]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

# Write your code here.

In [63]:
# TODO 4.1
# Run EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) on baseline_sql.
# Save the plan to reports/baseline_explain_analyze.txt.

explain_sql = f"EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) {baseline_sql}"

with engine.connect() as conn:
    plan_rows = conn.execute(text(explain_sql)).fetchall()

plan_text = "\n".join(row[0] for row in plan_rows)
Path("reports/baseline_explain_analyze.txt").write_text(plan_text, encoding="utf-8")

print(plan_text[:4000])

Sort  (cost=40185.48..40185.54 rows=22 width=212) (actual time=347.540..347.704 rows=22 loops=1)
  Sort Key: (count(l.listing_id)) DESC
  Sort Method: quicksort  Memory: 26kB
  Buffers: shared hit=15811 read=5288
  ->  GroupAggregate  (cost=39922.00..40184.99 rows=22 width=212) (actual time=341.466..347.652 rows=22 loops=1)
        Group Key: l.neighbourhood_id
        Buffers: shared hit=15811 read=5288
        ->  Sort  (cost=39922.00..39948.20 rows=10480 width=61) (actual time=340.786..341.671 rows=10480 loops=1)
              Sort Key: l.neighbourhood_id
              Sort Method: quicksort  Memory: 1072kB
              Buffers: shared hit=15811 read=5288
              ->  Hash Left Join  (cost=38872.34..39222.18 rows=10480 width=61) (actual time=325.622..336.991 rows=10480 loops=1)
                    Hash Cond: (l.listing_id = rc.listing_id)
                    Buffers: shared hit=15811 read=5288
                    ->  Hash Left Join  (cost=25145.41..25467.73 rows=10480 width=53

In [ ]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

# Write your code here.

In [65]:
# TODO 4.2
# Write reports/explain_notes.md with 3 specific observations from the plan.
# Do not write vague nonsense like 'the query is slow'.

explain_notes = f"""
# EXPLAIN notes

## Observation 1
The baseline query aggregates `core.calendar_day` for the first 30 days and `core.review` for all reviews before joining to `core.listing`.
This means the query still scans large base tables each time the dashboard query runs.

## Observation 2
The median calculation uses:
`PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price)`
which requires ordered work inside each neighbourhood group and is more expensive than simple aggregates like AVG or COUNT.

## Observation 3
Even though review and calendar metrics are pre-aggregated in CTEs, the full aggregation work is repeated on every execution of the baseline query.
This is a good candidate for a materialized view because dashboard users usually read summary data many times while source tables change less frequently.
""".strip()

Path("reports/explain_notes.md").write_text(explain_notes, encoding="utf-8")
print(explain_notes)

# EXPLAIN notes

## Observation 1
The baseline query aggregates `core.calendar_day` for the first 30 days and `core.review` for all reviews before joining to `core.listing`.
This means the query still scans large base tables each time the dashboard query runs.

## Observation 2
The median calculation uses:
`PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price)`
which requires ordered work inside each neighbourhood group and is more expensive than simple aggregates like AVG or COUNT.

## Observation 3
Even though review and calendar metrics are pre-aggregated in CTEs, the full aggregation work is repeated on every execution of the baseline query.
This is a good candidate for a materialized view because dashboard users usually read summary data many times while source tables change less frequently.


## 5. Create a materialized view

Metabase should read from a prepared object, not a fresh monster join.

In [ ]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f'''
-- write SQL here
'''
Path('sql/02_create_materialized_view.sql').write_text(optimized_sql)

In [67]:
# TODO 5.1
# Create optimized_sql.
# It should create student_<you>.mv_airbnb_neighbourhood_summary and at least two indexes.

optimized_sql = f"""
DROP MATERIALIZED VIEW IF EXISTS "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary;

CREATE MATERIALIZED VIEW "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary AS
WITH calendar_30 AS (
    SELECT
        cd.listing_id,
        AVG(cd.price) AS avg_calendar_price_30,
        ROUND(
            SUM(CASE WHEN cd.available = true THEN 1 ELSE 0 END)::numeric
            / NULLIF(COUNT(*), 0)::numeric,
            4
        ) AS availability_30_rate
    FROM core.calendar_day cd
    WHERE cd.date >= (SELECT MIN(date) FROM core.calendar_day)
      AND cd.date <  (SELECT MIN(date) FROM core.calendar_day) + INTERVAL '30 days'
    GROUP BY cd.listing_id
),
calendar_365 AS (
    SELECT
        cd.listing_id,
        ROUND(
            SUM(CASE WHEN cd.available = true THEN 1 ELSE 0 END)::numeric
            / NULLIF(COUNT(*), 0)::numeric,
            4
        ) AS availability_365_rate
    FROM core.calendar_day cd
    GROUP BY cd.listing_id
),
review_counts AS (
    SELECT
        r.listing_id,
        COUNT(*) AS total_reviews
    FROM core.review r
    GROUP BY r.listing_id
)
SELECT
    l.neighbourhood_id::text AS neighbourhood,
    COUNT(l.listing_id) AS num_listings,
    ROUND(AVG(l.listing_price), 2) AS avg_price,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY l.listing_price) AS median_price,
    ROUND(AVG(l.minimum_nights::numeric), 2) AS avg_minimum_nights,
    COALESCE(SUM(rc.total_reviews), 0) AS total_reviews,
    ROUND(
        COALESCE(SUM(rc.total_reviews), 0)::numeric
        / NULLIF(COUNT(l.listing_id), 0)::numeric,
        2
    ) AS reviews_per_listing,
    ROUND(AVG(c30.availability_30_rate), 4) AS availability_30_rate,
    ROUND(AVG(c365.availability_365_rate), 4) AS availability_365_rate
FROM core.listing l
LEFT JOIN calendar_30 c30
    ON c30.listing_id = l.listing_id
LEFT JOIN calendar_365 c365
    ON c365.listing_id = l.listing_id
LEFT JOIN review_counts rc
    ON rc.listing_id = l.listing_id
GROUP BY l.neighbourhood_id
ORDER BY num_listings DESC;

CREATE INDEX idx_{safe_student}_mv_airbnb_neighbourhood
    ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (neighbourhood);

CREATE INDEX idx_{safe_student}_mv_airbnb_num_listings
    ON "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary (num_listings DESC);
"""

Path("sql/02_create_materialized_view.sql").write_text(optimized_sql)
print(optimized_sql[:3000])


DROP MATERIALIZED VIEW IF EXISTS "student_nazanin_hesari".mv_airbnb_neighbourhood_summary;

CREATE MATERIALIZED VIEW "student_nazanin_hesari".mv_airbnb_neighbourhood_summary AS
WITH calendar_30 AS (
    SELECT
        cd.listing_id,
        AVG(cd.price) AS avg_calendar_price_30,
        ROUND(
            SUM(CASE WHEN cd.available = true THEN 1 ELSE 0 END)::numeric
            / NULLIF(COUNT(*), 0)::numeric,
            4
        ) AS availability_30_rate
    FROM core.calendar_day cd
    WHERE cd.date >= (SELECT MIN(date) FROM core.calendar_day)
      AND cd.date <  (SELECT MIN(date) FROM core.calendar_day) + INTERVAL '30 days'
    GROUP BY cd.listing_id
),
calendar_365 AS (
    SELECT
        cd.listing_id,
        ROUND(
            SUM(CASE WHEN cd.available = true THEN 1 ELSE 0 END)::numeric
            / NULLIF(COUNT(*), 0)::numeric,
            4
        ) AS availability_365_rate
    FROM core.calendar_day cd
    GROUP BY cd.listing_id
),
review_counts AS (
    SELECT
      

In [ ]:
# TODO 5.2
# Execute optimized_sql statement by statement.

# Write your code here.

In [69]:
# TODO 5.2
# Execute optimized_sql statement by statement.

statements = [s.strip() for s in optimized_sql.split(";") if s.strip()]

with engine.begin() as conn:
    conn.execute(text("SET statement_timeout = '300s'"))
    for i, stmt in enumerate(statements, start=1):
        print(f"[{i}/{len(statements)}] running...")
        conn.execute(text(stmt))
        print("done")

[1/4] running...
done
[2/4] running...
done
[3/4] running...
done
[4/4] running...
done


In [71]:
check_sql = f'''select * from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary order by num_listings desc limit 10;'''
pd.read_sql(check_sql, engine)

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,total_reviews,reviews_per_listing,availability_30_rate,availability_365_rate
0,22,1808,271.28,240.0,3.95,62753.0,34.71,0.1554,0.2256
1,2,1207,315.88,245.5,4.01,106496.0,88.23,0.2189,0.3238
2,20,1199,280.47,250.0,5.46,45931.0,38.31,0.1703,0.2662
3,1,923,307.72,240.0,4.89,76899.0,83.31,0.2214,0.3641
4,4,736,255.04,214.5,4.24,26668.0,36.23,0.1390,0.2199
5,21,735,309.58,238.0,3.99,29444.0,40.06,0.1927,0.2690
6,13,654,251.66,222.0,4.06,20448.0,31.27,0.1440,0.2171
7,14,547,204.67,184.0,6.08,16461.0,30.09,0.1286,0.1832
8,18,485,370.50,196.5,3.34,22623.0,46.65,0.1644,0.2178
9,3,436,216.60,195.0,4.40,13988.0,32.08,0.1266,0.2042


## 6. Compare latency

Numbers or it did not happen.

In [73]:
dashboard_sql = f'''
select neighbourhood, num_listings, avg_price, median_price,
       total_reviews, reviews_per_listing, availability_30_rate, availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''
dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)
perf = pd.DataFrame([
    {'query': 'baseline_direct_query', 'best_seconds': min(baseline_times), 'avg_seconds': sum(baseline_times)/len(baseline_times)},
    {'query': 'materialized_view_read', 'best_seconds': min(dashboard_times), 'avg_seconds': sum(dashboard_times)/len(dashboard_times)},
])
perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.448048,0.472778,1.000000
1,materialized_view_read,0.180243,0.201223,2.485793


In [75]:
# TODO 6
# Compare latency

dashboard_sql = f'''
select
    neighbourhood,
    num_listings,
    avg_price,
    median_price,
    total_reviews,
    reviews_per_listing,
    availability_30_rate,
    availability_365_rate
from "{STUDENT_SCHEMA}".mv_airbnb_neighbourhood_summary
order by num_listings desc;
'''

dashboard_df, dashboard_times = timed_read_sql(dashboard_sql, repeats=5)

perf = pd.DataFrame([
    {
        'query': 'baseline_direct_query',
        'best_seconds': min(baseline_times),
        'avg_seconds': sum(baseline_times) / len(baseline_times)
    },
    {
        'query': 'materialized_view_read',
        'best_seconds': min(dashboard_times),
        'avg_seconds': sum(dashboard_times) / len(dashboard_times)
    },
])

perf['speedup_vs_baseline_best'] = perf.loc[0, 'best_seconds'] / perf['best_seconds']

display(dashboard_df.head())
display(perf)

,neighbourhood,num_listings,avg_price,median_price,total_reviews,reviews_per_listing,availability_30_rate,availability_365_rate
0,22,1808,271.28,240.0,62753.0,34.71,0.1554,0.2256
1,2,1207,315.88,245.5,106496.0,88.23,0.2189,0.3238
2,20,1199,280.47,250.0,45931.0,38.31,0.1703,0.2662
3,1,923,307.72,240.0,76899.0,83.31,0.2214,0.3641
4,4,736,255.04,214.5,26668.0,36.23,0.1390,0.2199


,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.448048,0.472778,1.000000
1,materialized_view_read,0.177823,0.198372,2.519626


## 7. Metabase dashboard

Open the shared Metabase URL and create:

```text
QBC12 HW01 - <your-github-username> - Airbnb Ops
```

Required cards:

1. listings by neighbourhood
2. average price by neighbourhood
3. review activity by neighbourhood
4. availability rate by neighbourhood
5. top neighbourhoods table

Screenshot path:

```text
screenshots/metabase_dashboard.png
```

In [ ]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

# Write your code here.

In [77]:
# TODO 7.1
# Write reports/hw01_b_sql_performance.md.
# Include schema, runtimes, speedup, what changed, and Metabase screenshot/link.

baseline_best = min(baseline_times)
baseline_avg = sum(baseline_times) / len(baseline_times)
mv_best = min(dashboard_times)
mv_avg = sum(dashboard_times) / len(dashboard_times)
speedup_best = baseline_best / mv_best if mv_best else None

report_md = f"""
# HW01-B — SQL Performance and Metabase

## Student schema
- schema: `{STUDENT_SCHEMA}`

## Baseline query
The baseline query reads directly from base tables in `core`:
- `core.listing`
- `core.calendar_day`
- `core.review`

It computes:
- listing count by neighbourhood
- average price
- median price
- average minimum nights
- total reviews
- reviews per listing
- 30-day availability rate

## Optimization approach
I created a materialized view:

- `{STUDENT_SCHEMA}.mv_airbnb_neighbourhood_summary`

This moves repeated aggregation work out of dashboard runtime and into a prepared database object.

### What changed
1. Precomputed the neighbourhood summary into a materialized view.
2. Added `availability_365_rate` so dashboard reads do not need extra aggregation.
3. Added indexes on:
   - `neighbourhood`
   - `num_listings DESC`

## Runtime comparison

### Baseline direct query
- runs: {baseline_times}
- best_seconds: {baseline_best:.4f}
- avg_seconds: {baseline_avg:.4f}

### Materialized view read
- runs: {dashboard_times}
- best_seconds: {mv_best:.4f}
- avg_seconds: {mv_avg:.4f}

### Speedup
- speedup_vs_baseline_best: {speedup_best:.4f}x

## Explain-plan summary
- The baseline query performs repeated aggregation on large source tables.
- Median calculation with `PERCENTILE_CONT` adds extra sorting/ordering work inside groups.
- The materialized view reduces repeated work at dashboard read time.

## Metabase dashboard
Dashboard name:
- `QBC12 HW01 - <your-github-username> - Airbnb Ops`

Required screenshot path:
- `screenshots/metabase_dashboard.png`

Metabase link:
- Paste your Metabase dashboard URL here

## Notes
Because `core.listing` does not expose a text `neighbourhood` column in the shared schema I used `neighbourhood_id::text AS neighbourhood` in the output.
"""

Path("reports/hw01_b_sql_performance.md").write_text(report_md.strip(), encoding="utf-8")
print(report_md)


# HW01-B — SQL Performance and Metabase

## Student schema
- schema: `student_nazanin_hesari`

## Baseline query
The baseline query reads directly from base tables in `core`:
- `core.listing`
- `core.calendar_day`
- `core.review`

It computes:
- listing count by neighbourhood
- average price
- median price
- average minimum nights
- total reviews
- reviews per listing
- 30-day availability rate

## Optimization approach
I created a materialized view:

- `student_nazanin_hesari.mv_airbnb_neighbourhood_summary`

This moves repeated aggregation work out of dashboard runtime and into a prepared database object.

### What changed
1. Precomputed the neighbourhood summary into a materialized view.
2. Added `availability_365_rate` so dashboard reads do not need extra aggregation.
3. Added indexes on:
   - `neighbourhood`
   - `num_listings DESC`

## Runtime comparison

### Baseline direct query
- runs: [0.4480480000056559, 0.48890949999622535, 0.4813768999883905]
- best_seconds: 0.4480
- avg_

In [79]:
for file in ['sql/01_baseline_neighbourhood_summary.sql','sql/02_create_materialized_view.sql','reports/baseline_explain_analyze.txt','reports/explain_notes.md','reports/hw01_b_sql_performance.md']:
    assert Path(file).exists(), f'Missing {file}'
assert len(dashboard_df) > 0
perf

,query,best_seconds,avg_seconds,speedup_vs_baseline_best
0,baseline_direct_query,0.448048,0.472778,1.000000
1,materialized_view_read,0.177823,0.198372,2.519626


## Commit

```bash
git add sql reports screenshots notebooks
git commit -m "HW01-B SQL performance and Metabase dashboard"
```